# Práctica 2 — RNA-seq: Control de Calidad, Mapeo y Pseudoalineamiento
## El papel de **UPF1** en células madre embrionarias humanas (GSE81214)

**Máster en Bioinformática — Módulo de Transcriptómica**

**Contexto:** el *Nonsense-Mediated RNA Decay* (NMD) degrada ARNm con codones de
terminación prematuros. **UPF1**, su proteína central, regula además genes de
pluripotencia y diferenciación en hESCs. En esta práctica analizamos el impacto
transcriptómico del *knockdown* de UPF1 (siRNA) frente a un siRNA control.

**Diseño:** 2 réplicas control (SRR3480371, SRR3480372) vs 2 réplicas siRNA-UPF1
(SRR3480373, SRR3480374) · Illumina HiSeq 2500 · 50 bp single-end *stranded*.

> **Datos de clase:** para que el pipeline quepa en una sesión de Colab trabajamos con
> **submuestras** de las lecturas originales. El procedimiento es idéntico al análisis
> de los datos completos.

| Parte | Contenido | Herramientas |
|---|---|---|
| I | Preparación y control de calidad | FastQC |
| II | Alineamiento al genoma (chr19) | STAR, samtools |
| III | Cuantificación génica | featureCounts |
| IV | Pseudoalineamiento | Salmon + pymportx |
| V | Comparación de métodos | pandas, matplotlib |

> ⏱️ La celda de instalación puede tardar **5-10 minutos** (se descargan e instalan
> varias herramientas bioinformáticas). Ejecuta **Entorno de ejecución → Ejecutar todas**
> y espera a que termine.

## 0. Instalación de herramientas

Instalamos:

- **FastQC** y **samtools** desde los repositorios de Ubuntu.
- **STAR** (alineador), **featureCounts** (paquete subread), **Salmon** (pseudoalineador)
  y **gffread** (extracción de secuencias de transcritos) como binarios precompilados.
- **pymportx** desde PyPI: adaptación en Python de `tximport` para agregar cuantificaciones
  de transcritos a nivel de gen.

In [ ]:
# ============================================================================
# CELDA 0: Instalación de herramientas bioinformáticas (5-10 min, una vez por sesión)
# ============================================================================
import os, glob, subprocess

# --- 1. FastQC, samtools y featureCounts (paquete subread) desde apt ---
!apt-get install -y -qq fastqc samtools subread > /dev/null 2>&1

# --- 2. STAR (binario estático) ---
!wget -q https://github.com/alexdobin/STAR/releases/download/2.7.11b/STAR_2.7.11b.zip -O /tmp/star.zip
!unzip -q -o /tmp/star.zip -d /tmp/star
STAR = "/tmp/star/STAR_2.7.11b/Linux_x86_64_static/STAR"

# --- 3. Salmon (binario precompilado) ---
!wget -q https://github.com/COMBINE-lab/salmon/releases/download/v2.9.1/salmon-cli-x86_64-unknown-linux-gnu.tar.xz -O /tmp/salmon.tar.xz
!mkdir -p /tmp/salmon && tar -xJf /tmp/salmon.tar.xz -C /tmp/salmon
SALMON = (glob.glob("/tmp/salmon/**/salmon", recursive=True) or [None])[0]

# --- 4. gffread (extraer secuencias de transcritos del genoma) ---
!wget -q https://github.com/gpertea/gffread/releases/download/v0.12.9/gffread-0.12.9.Linux_x86_64.tar.gz -O /tmp/gffread.tar.gz
!mkdir -p /tmp/gffread && tar -xzf /tmp/gffread.tar.gz -C /tmp/gffread
GFFREAD = (glob.glob("/tmp/gffread/**/gffread", recursive=True) or [None])[0]

# --- 5. pymportx (Python, desde PyPI; pandas fijado a la versión de Colab) ---
!pip install -q pymportx "pandas==2.2.3"

# --- Verificación (robusta: no se rompe aunque una herramienta falle) ---
FC = "featureCounts"  # instalado vía apt (paquete subread)
herramientas = {"fastqc": "fastqc", "samtools": "samtools", "STAR": STAR,
                "featureCounts": FC, "salmon": SALMON, "gffread": GFFREAD}
for nombre, ruta in herramientas.items():
    if ruta is None:
        print(f"✗ {nombre}: NO ENCONTRADO")
        continue
    try:
        r = subprocess.run(f"{ruta} --version", shell=True, capture_output=True,
                           text=True, errors="replace")
        version = (r.stdout or r.stderr).strip().split("\n")[0][:60]
        print(f"✓ {nombre}: {version}")
    except Exception as e:
        print(f"✗ {nombre}: FALLO ({e})")

import pymportx
print("✓ pymportx instalado correctamente")

## 1. Datos de la práctica

Descargamos las 4 submuestras FASTQ desde el repositorio del curso. Si trabajas con tus
propios archivos, cambia `MODO` a `"SUBIR"` o `"DRIVE"` (igual que en la práctica 1).

In [ ]:
# ============================================================================
# CELDA 1: Obtención de los datos
# ============================================================================
from pathlib import Path
import urllib.request

MODO = "URL"   # "URL" | "SUBIR" | "DRIVE"

# Muestras del experimento GSE81214
MUESTRAS = {
    "SRR3480371": "Control_1",
    "SRR3480372": "Control_2",
    "SRR3480373": "Tratamiento_1",
    "SRR3480374": "Tratamiento_2",
}

DATA_DIR = Path("/content/datos_rnaseq")

if MODO == "URL":
    DATA_DIR.mkdir(exist_ok=True)
    URL_BASE = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/02-rnaseq-upf1/datos/"
    for srr in MUESTRAS:
        destino = DATA_DIR / f"{srr}.subset.fastq.gz"
        if not destino.exists():
            print(f"Descargando {srr}...")
            urllib.request.urlretrieve(URL_BASE + f"{srr}.subset.fastq.gz", destino)
    # Anotación GTF (GENCODE v38, subconjunto de genes del chr19)
    gtf_dest = DATA_DIR / "subset_gencode.v38.annotation.gtf"
    if not gtf_dest.exists():
        urllib.request.urlretrieve(URL_BASE + "subset_gencode.v38.annotation.gtf", gtf_dest)

elif MODO == "SUBIR":
    from google.colab import files
    DATA_DIR = Path("/content")
    print("Sube los 4 archivos .subset.fastq.gz:")
    files.upload()

elif MODO == "DRIVE":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/RNAseq_UPF1")  # <-- AJUSTA ESTA RUTA

FASTQS = {srr: DATA_DIR / f"{srr}.subset.fastq.gz" for srr in MUESTRAS}
for srr, f in FASTQS.items():
    assert f.exists(), f"No se encuentra {f}"
    print(f"✓ {f.name}: {f.stat().st_size / 1e3:.0f} KB")

GTF_FILE = DATA_DIR / "subset_gencode.v38.annotation.gtf"
assert GTF_FILE.exists(), f"No se encuentra {GTF_FILE}"
print(f"✓ {GTF_FILE.name} (anotación GENCODE v38)")

# Directorio de salida para todos los resultados
OUT_DIR = Path("/content/rnaseq_UPF1_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"\n✓ Resultados se guardarán en: {OUT_DIR}")

---
# PARTE I — Preparación y Control de Calidad

## Actividad 1.1: Descarga y preparación de datos

En un análisis real descargaríamos los FASTQ completos desde SRA con `fasterq-dump`:

```bash
fasterq-dump SRR3480371 --split-files --threads 4
```

En esta práctica usamos submuestras ya preparadas, pero primero vamos a **examinar el
formato FASTQ** con las primeras líneas de un archivo.

In [ ]:
# ============================================================================
# CELDA 2: Examinar el formato FASTQ
# ============================================================================

# Las 4 primeras lecturas del archivo (cada lectura ocupa 4 líneas)
!zcat /content/datos_rnaseq/SRR3480371.subset.fastq.gz | head -16

# Contar el número de lecturas de cada muestra (líneas totales / 4)
print("\nNúmero de lecturas por muestra:")
import subprocess
for srr, cond in MUESTRAS.items():
    n = subprocess.run(f"zcat {FASTQS[srr]} | wc -l", shell=True,
                       capture_output=True, text=True).stdout.strip()
    print(f"  {srr} ({cond}): {int(n)//4:,} lecturas")

> ❓ **Pregunta 1.1:** ¿Qué información contiene cada una de las 4 líneas del formato
> FASTQ? Fíjate en el identificador, la secuencia y los caracteres de calidad.

## Actividad 1.2: Control de calidad con FastQC

Ejecutamos FastQC sobre las 4 muestras. FastQC genera un informe HTML por muestra y un
archivo `fastqc_data.txt` con los datos numéricos, que parseamos aquí para comparar.

In [ ]:
# ============================================================================
# CELDA 3: FastQC sobre todas las muestras
# ============================================================================
import subprocess

QC_DIR = OUT_DIR / "fastqc"
QC_DIR.mkdir(exist_ok=True)

for srr, fq in FASTQS.items():
    print(f"Analizando {srr}...")
    subprocess.run(f"fastqc {fq} -o {QC_DIR} -t 4", shell=True,
                   capture_output=True, text=True)

print("\n✓ Informes generados:")
for f in sorted(QC_DIR.glob("*.html")):
    print(" ", f.name)

In [ ]:
# ============================================================================
# CELDA 4: Comparación de métricas clave de FastQC
# ============================================================================
import re, zipfile
import pandas as pd

def parse_fastqc(zip_path):
    """Extrae métricas básicas del fastqc_data.txt de un informe FastQC."""
    with zipfile.ZipFile(zip_path) as z:
        data = z.read([n for n in z.namelist() if n.endswith("fastqc_data.txt")][0]).decode(errors="replace")
    def campo(nombre):
        m = re.search(rf"^{re.escape(nombre)}\t(\S+)", data, re.M)
        return m.group(1) if m else "?"
    m = {"Secuencias": campo("Total Sequences"),
         "Longitud (bp)": campo("Sequence length"),
         "%GC": campo("%GC")}
    # Estado de los módulos clave (pass / warn / fail)
    for mod in ["Per base sequence quality", "Per base sequence content",
                "Adapter Content", "Overrepresented sequences"]:
        hit = re.search(rf">{re.escape(mod)}\t(\w+)", data)
        m[mod] = hit.group(1) if hit else "?"
    return m

tabla_qc = {}
for z in sorted(QC_DIR.glob("*_fastqc.zip")):
    srr = z.name.split(".")[0]
    tabla_qc[srr] = parse_fastqc(z)

df_qc = pd.DataFrame(tabla_qc).T
df_qc["Condición"] = [MUESTRAS[s] for s in df_qc.index]
display(df_qc)

In [ ]:
# ============================================================================
# CELDA 4b: Gráficos clave de FastQC
# ============================================================================
# En Colab no se puede mostrar el informe HTML local directamente;
# extraemos los gráficos PNG del ZIP de FastQC y los mostramos aquí.
import io
from PIL import Image
from IPython.display import display

def mostrar_grafico_fastqc(nombre_png, titulo):
    """Muestra un gráfico de FastQC de las 4 muestras, lado a lado."""
    imagenes, etiquetas = [], []
    for z in sorted(QC_DIR.glob("*_fastqc.zip")):
        srr = z.name.split(".")[0]
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith(f"Images/{nombre_png}")]
            if hits:
                imagenes.append(Image.open(io.BytesIO(zf.read(hits[0]))))
                etiquetas.append(srr)
    if not imagenes:
        print(f"(sin gráfico {nombre_png})")
        return
    w = sum(im.width for im in imagenes)
    h = max(im.height for im in imagenes)
    combo = Image.new("RGB", (w, h), "white")
    x = 0
    for im in imagenes:
        combo.paste(im, (x, 0))
        x += im.width
    print(f"=== {titulo} | {", ".join(etiquetas)} (izquierda → derecha) ===")
    display(combo)

for png, titulo in [("per_base_quality.png", "Calidad por posición"),
                    ("per_base_sequence_content.png", "Contenido de bases por posición"),
                    ("adapter_content.png", "Contenido de adaptadores")]:
    mostrar_grafico_fastqc(png, titulo)

> ❓ **Preguntas 1.2:**
> 1. ¿Cuál es la calidad promedio de las lecturas?
> 2. ¿Observas algún sesgo en la composición de bases? *(Pista: es normal ver desviación en
>    los primeros ~10 nt en librerías RNA-seq por el cebado con hexámeros aleatorios.)*
> 3. ¿Hay presencia de adaptadores o contaminantes?
> 4. ¿Recomendarías algún filtrado o *trimming* adicional?

---
# PARTE II — Alineamiento y Mapeo con STAR

## Actividad 2.1: Preparación del genoma de referencia (cromosoma 19)

Descargamos de **Ensembl** la secuencia del cromosoma 19 y su anotación GTF, y generamos
el índice de STAR. Trabajar con un solo cromosoma reduce el tiempo de indexación de
~30 min a ~1 minuto, manteniendo intacta toda la metodología.

In [ ]:
# ============================================================================
# CELDA 5: Descarga del genoma (chr19) y preparación de la anotación
# ============================================================================
REF_DIR = Path("/content/referencia")
REF_DIR.mkdir(exist_ok=True)

# OJO con la nomenclatura: el GTF de GENCODE usa "chr19" (estilo UCSC),
# mientras que Ensembl usa "19". Si no coinciden, STAR no encuentra los exones.
# Descargamos el chr19 de UCSC (hg38 = GRCh38; la secuencia del cromosoma
# principal es idéntica a la de GRCh38.p13 de GENCODE v38).
!wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr19.fa.gz -O {REF_DIR}/chr19.fa.gz
!gunzip -kf {REF_DIR}/chr19.fa.gz

# Anotación GTF: usamos el fichero GENCODE v38 proporcionado en los datos
import shutil
shutil.copy(GTF_FILE, REF_DIR / "chr19.gtf")

# Comprobación de coherencia: el nombre del cromosoma debe coincidir
nombre_fasta = open(REF_DIR / "chr19.fa").readline().strip().lstrip(">")
with open(REF_DIR / "chr19.gtf") as f:
    for linea in f:
        if not linea.startswith("#"):
            nombre_gtf = linea.split("\t")[0]
            break
print(f"✓ Cromosoma en FASTA: '{nombre_fasta}' | en GTF: '{nombre_gtf}'")
assert nombre_fasta == nombre_gtf, "¡Los nombres de cromosoma no coinciden!"

import re
genes = {re.search(r'gene_name "([^"]+)"', l).group(1)
         for l in open(REF_DIR / "chr19.gtf")
         if "\tgene\t" in l and 'gene_name "' in l}
print(f"✓ GTF preparado: {len(genes)} genes anotados: {", ".join(sorted(genes))}")

In [ ]:
# ============================================================================
# CELDA 6: Generación del índice de STAR
# ============================================================================
import subprocess

STAR = "/tmp/star/STAR_2.7.11b/Linux_x86_64_static/STAR"
INDEX_DIR = "/content/referencia/STAR_index"

cmd = f"""{STAR} --runMode genomeGenerate \
  --genomeDir {INDEX_DIR} \
  --genomeFastaFiles {REF_DIR}/chr19.fa \
  --sjdbGTFfile {REF_DIR}/chr19.gtf \
  --sjdbOverhang 49 \
  --genomeSAindexNbases 11 \
  --runThreadN 2"""

# --sjdbOverhang = longitud_lectura - 1 (50 bp → 49)
# --genomeSAindexNbases 11: ajuste necesario para genomas pequeños como chr19
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("✓ Índice de STAR generado" if r.returncode == 0 else r.stderr[-2000:])
!ls -lh {INDEX_DIR}/ | head -8

> ❓ **Pregunta 2.1:** ¿Por qué es importante incluir la anotación GTF durante la
> indexación? *(Pista: piensa en qué información aporta `--sjdbGTFfile` sobre las
> uniones exón-exón y cómo afecta eso al mapeo de lecturas que cruzan *splice junctions*.)*

## Actividad 2.2: Alineamiento con STAR

Alineamos cada muestra al índice generado. STAR produce BAM **ordenados por coordenadas**
(`--outSAMtype BAM SortedByCoordinate`) y un informe `Log.final.out` por muestra.

In [ ]:
# ============================================================================
# CELDA 7: Alineamiento de las 4 muestras con STAR
# ============================================================================
import time

STAR_DIR = OUT_DIR / "STAR"
STAR_DIR.mkdir(exist_ok=True)

tiempos_star = {}
for srr, cond in MUESTRAS.items():
    t0 = time.time()
    cmd = f"""{STAR} --runThreadN 2 \
      --genomeDir {INDEX_DIR} \
      --readFilesIn {FASTQS[srr]} --readFilesCommand zcat \
      --outFileNamePrefix {STAR_DIR}/{srr}_ \
      --outSAMtype BAM SortedByCoordinate"""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tiempos_star[srr] = time.time() - t0
    estado = "✓" if r.returncode == 0 else "✗ ERROR"
    print(f"{estado} {srr} ({cond}): {tiempos_star[srr]:.0f} s")

In [ ]:
# ============================================================================
# CELDA 8: Análisis de los Log.final.out
# ============================================================================

def parse_star_log(path):
    """Extrae las métricas clave de un Log.final.out de STAR."""
    m = {}
    with open(path) as f:
        for linea in f:
            if "%" in linea or "Number of input reads" in linea:
                partes = [p.strip() for p in linea.split("|")]
                if len(partes) == 2:
                    m[partes[0]] = partes[1]
    return m

tabla_star = {}
for srr in MUESTRAS:
    tabla_star[srr] = parse_star_log(STAR_DIR / f"{srr}_Log.final.out")

metricas = ["Number of input reads",
            "Uniquely mapped reads %",
            "% of reads mapped to multiple loci",
            "% of reads unmapped: too short",
            "% of chimeric reads"]
df_star = pd.DataFrame(tabla_star).T
cols = [m for m in metricas if m in df_star.columns]
df_star["Condición"] = [MUESTRAS[s] for s in df_star.index]
df_star["Tiempo (s)"] = [round(tiempos_star[s]) for s in df_star.index]
display(df_star[cols + ["Condición", "Tiempo (s)"]])

> ❓ **Preguntas 2.2:**
> 1. ¿Cuál es el porcentaje de lecturas mapeadas en cada muestra?
> 2. ¿Hay diferencias significativas en las tasas de mapeo entre condiciones?
> 3. ¿Qué porcentaje de lecturas se mapea a regiones con *splice junctions*?
>    *(Pista: revisa también el archivo `SJ.out.tab` de una muestra.)*

## Actividad 2.3: Calidad del alineamiento — indexación de BAM

Indexamos los BAM con samtools para poder consultarlos eficientemente (p. ej. en IGV).

In [ ]:
# ============================================================================
# CELDA 9: Indexación y estadísticas de los BAM
# ============================================================================
for srr in MUESTRAS:
    bam = STAR_DIR / f"{srr}_Aligned.sortedByCoord.out.bam"
    subprocess.run(f"samtools index {bam}", shell=True, check=True)
    print(f"✓ {srr} indexado")

# Estadísticas rápidas de un BAM de ejemplo
print("\n--- flagstat de SRR3480371 (Control 1) ---")
!samtools flagstat {STAR_DIR}/SRR3480371_Aligned.sortedByCoord.out.bam

> ❓ **Pregunta 2.3:** ¿Por qué es útil que STAR genere los BAM ordenados por
> coordenadas directamente? ¿Qué pasos posteriores lo requieren?

---
# PARTE III — Cuantificación de la Expresión Génica con featureCounts

## Actividad 3.1: Conteo de lecturas por gen

featureCounts asigna las lecturas de los BAM a los genes de la anotación GTF.
**Importante:** la librería es *stranded*, así que indicamos la orientación con `-s`.
Si los resultados muestran un % de asignación muy bajo, prueba a cambiar `-s 2` por `-s 1`:
es un ejercicio excelente para entender la importancia de conocer el protocolo de librería.

In [ ]:
# ============================================================================
# CELDA 10: Cuantificación con featureCounts
# ============================================================================
# FC, SALMON, STAR y GFFREAD quedaron definidos en la CELDA 0
bams = " ".join(str(STAR_DIR / f"{s}_Aligned.sortedByCoord.out.bam") for s in MUESTRAS)

# -s 2: librería stranded reversa (típica de protocolos dUTP). Ver Pregunta 3.1.4.
r = subprocess.run(f"{FC} -T 4 -s 2 -a {REF_DIR}/chr19.gtf -o {OUT_DIR}/gene_counts.txt {bams}",
                   shell=True, capture_output=True, text=True)
print(r.stderr[-800:] if r.returncode != 0 else "✓ Cuantificación completada")

# Primeras líneas del archivo de conteos
!head -12 {OUT_DIR}/gene_counts.txt

In [ ]:
# ============================================================================
# CELDA 11: Resumen de la asignación
# ============================================================================
import pandas as pd

# El resumen (.summary) muestra cuántas lecturas se asignan y por qué se descartan
resumen = pd.read_csv(str(OUT_DIR / "gene_counts.txt.summary"), sep="\t", index_col=0)
resumen.columns = [c.split("/")[-1].replace("_Aligned.sortedByCoord.out.bam", "") for c in resumen.columns]
display(resumen)

asignadas = resumen.loc["Assigned"].astype(int)
total = resumen.loc[[i for i in resumen.index if i != "Assigned"]].astype(int).sum() + asignadas
print("\nPorcentaje de lecturas asignadas a genes:")
for srr in MUESTRAS:
    print(f"  {srr} ({MUESTRAS[srr]}): {100*asignadas[srr]/total[srr]:.1f}%")

> ❓ **Preguntas 3.1:**
> 1. ¿Qué información contienen las primeras 6 columnas de `gene_counts.txt`?
> 2. ¿Cuántos genes se detectaron en total?
> 3. ¿Qué porcentaje de lecturas se asignó exitosamente a genes?
> 4. Repite la cuantificación cambiando `-s 2` por `-s 1` o `-s 0`. ¿Cómo cambia el %
>    de asignación? ¿Qué te dice esto sobre la orientación real de la librería?

## Actividad 3.2: Exploración de los datos de conteo

In [ ]:
# ============================================================================
# CELDA 12: Exploración de la matriz de conteos
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np

counts_fc = pd.read_csv(OUT_DIR / "gene_counts.txt", sep="\t", skiprows=1, index_col=0)
counts_fc = counts_fc.iloc[:, 4:]  # columnas de muestras (tras Geneid, Chr, Start, End, Strand, Length)
counts_fc.columns = [c.split("/")[-1].replace("_Aligned.sortedByCoord.out.bam", "") for c in counts_fc.columns]

print(f"Genes en la matriz: {counts_fc.shape[0]}")
print("\nEstadísticas por muestra:")
display(counts_fc.describe().loc[["mean", "50%", "max"]].round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Distribución de conteos por gen (escala log)
axes[0].hist(np.log10(counts_fc.sum(axis=1) + 1), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("log10(conteos totales + 1)")
axes[0].set_ylabel("Nº de genes")
axes[0].set_title("Distribución de conteos por gen")

# Boxplot por muestra (log)
counts_fc.assign(**{}).apply(lambda x: np.log10(x + 1)).boxplot(ax=axes[1])
axes[1].set_ylabel("log10(conteos + 1)")
axes[1].set_title("Distribución por muestra")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig(OUT_DIR / "exploracion_conteos.png", dpi=150)
plt.show()

> ❓ **Preguntas 3.2:**
> 1. ¿Cómo es la distribución de conteos por gen? ¿Es simétrica? ¿Qué implica esto para
>    los análisis estadísticos posteriores?
> 2. ¿Hay diferencias evidentes entre las muestras control y tratamiento a este nivel?

---
# PARTE IV — Cuantificación Alternativa con Salmon

## Actividad 4.1: Preparación del índice del transcriptoma

En lugar de alinear al genoma, Salmon cuantifica directamente contra las **secuencias de
transcritos**. Las extraemos del chr19 con `gffread` (genoma + GTF → FASTA de transcritos)
y generamos el índice de Salmon.

In [ ]:
# ============================================================================
# CELDA 13: Extracción de transcritos e índice de Salmon
# ============================================================================
# GFFREAD y SALMON quedaron definidos en la CELDA 0

# Extraer las secuencias de los transcritos del chr19
!{GFFREAD} -w {REF_DIR}/chr19_transcripts.fa -g {REF_DIR}/chr19.fa {REF_DIR}/chr19.gtf
!grep -c ">" {REF_DIR}/chr19_transcripts.fa

# Generar el índice de Salmon (-k 25 es adecuado para lecturas de 50 bp)
r = subprocess.run(f"{SALMON} index -t {REF_DIR}/chr19_transcripts.fa -i {REF_DIR}/salmon_index -k 25",
                   shell=True, capture_output=True, text=True)
print("✓ Índice de Salmon generado" if r.returncode == 0 else r.stderr[-1500:])

> ❓ **Pregunta 4.1:** ¿Cuáles son las ventajas de la cuantificación a nivel de
> transcrito frente al alineamiento genómico? Piensa en velocidad, uso de memoria,
> genes con pseudogenes o lecturas multimapeo, e isoformas.

## Actividad 4.2: Cuantificación con Salmon (quasi-mapping)

Ejecutamos Salmon en modo quasi-mapping para cada muestra. `-l A` detecta automáticamente
el tipo de librería (orientación, single/paired-end).

In [ ]:
# ============================================================================
# CELDA 14: Cuantificación con Salmon
# ============================================================================
import time, json as _json

SALMON_DIR = OUT_DIR / "salmon"
SALMON_DIR.mkdir(exist_ok=True)

tiempos_salmon = {}
for srr, cond in MUESTRAS.items():
    t0 = time.time()
    r = subprocess.run(
        f"{SALMON} quant -i {REF_DIR}/salmon_index -l A -r {FASTQS[srr]} "
        f"--validateMappings -o {SALMON_DIR}/{srr} -p 2",
        shell=True, capture_output=True, text=True)
    tiempos_salmon[srr] = time.time() - t0
    print(("✓" if r.returncode == 0 else "✗ ERROR") + f" {srr} ({cond}): {tiempos_salmon[srr]:.0f} s")

# Tasas de mapeo desde meta_info.json
print("\nTasa de mapeo por muestra:")
for srr in MUESTRAS:
    with open(SALMON_DIR / srr / "aux_info" / "meta_info.json") as f:
        meta = _json.load(f)
    print(f"  {srr} ({MUESTRAS[srr]}): {meta['percent_mapped']:.1f}% mapeadas "
          f"(librería detectada: {meta['library_types']})")

In [ ]:
# ============================================================================
# CELDA 15: Examinar un archivo quant.sf
# ============================================================================
quant = pd.read_csv(SALMON_DIR / "SRR3480371" / "quant.sf", sep="\t")
print("Columnas:", quant.columns.tolist())
display(quant.head())

# Comparación de tiempos STAR vs Salmon
comparacion_tiempos = pd.DataFrame({
    "STAR (s)": pd.Series(tiempos_star).round(0),
    "Salmon (s)": pd.Series(tiempos_salmon).round(0),
})
display(comparacion_tiempos)

> ❓ **Preguntas 4.2:**
> 1. ¿Cuál es el porcentaje de lecturas mapeadas en cada muestra?
> 2. ¿Hay diferencias en las tasas de mapeo comparado con STAR? ¿A qué pueden deberse?
> 3. ¿Cómo se comparan los tiempos de ejecución entre STAR y Salmon? ¿Qué ventaja
>    práctica tiene esto con datasets grandes?

## Actividad 4.3: De transcritos a genes con pymportx

Salmon cuantifica **transcritos** (isoformas). Para el análisis de expresión diferencial
a nivel génico agregamos con **pymportx** (la adaptación Python de `tximport`), que
pondera correctamente por la longitud y abundancia de cada transcrito.

In [ ]:
# ============================================================================
# CELDA 16: Tabla tx2gene a partir del GTF
# ============================================================================
import re

# Extraer pares transcript_id → gene_id de las líneas de transcrito del GTF
pares = []
with open(REF_DIR / "chr19.gtf") as f:
    for linea in f:
        if linea.startswith("#"):
            continue
        campos = linea.split("\t")
        if len(campos) > 8 and campos[2] == "transcript":
            t = re.search(r'transcript_id "([^"]+)"', campos[8])
            g = re.search(r'gene_id "([^"]+)"', campos[8])
            if t and g:
                pares.append((t.group(1).split(".")[0], g.group(1).split(".")[0]))

tx2gene = pd.DataFrame(pares, columns=["transcript_id", "gene_id"]).drop_duplicates()
# pymportx lee este CSV con cabecera; usaremos ignoreTxVersion=True en la CELDA 17
tx2gene.to_csv(OUT_DIR / "tx2gene.csv", index=False)
print(f"✓ tx2gene: {len(tx2gene)} transcritos → {tx2gene['gene_id'].nunique()} genes")
display(tx2gene.head())

In [ ]:
# ============================================================================
# CELDA 17: Agregación a nivel de gen con pymportx
# ============================================================================
from pymportx import salmon as pmx_salmon

# pymportx espera los quant.sf COMPRIMIDOS (quant.sf.gz); Salmon los genera sin comprimir
import subprocess
for srr in MUESTRAS:
    q = SALMON_DIR / srr / "quant.sf"
    qz = SALMON_DIR / srr / "quant.sf.gz"
    if q.exists() and not qz.exists():
        subprocess.run(f"gzip -k {q}", shell=True, check=True)

carpetas = [str(SALMON_DIR / s) for s in MUESTRAS]
counts_salmon = pmx_salmon.read_salmon(carpetas,
                                       tx_out=False,                 # agregar a nivel de gen
                                       tx2gene=str(OUT_DIR / "tx2gene.csv"),
                                       countsFromAbundance="no",
                                       ignoreTxVersion=True)         # ENST... sin sufijo de versión

# read_salmon devuelve un objeto AnnData: muestras en filas (obs), genes en columnas (var)
counts_salmon_df = pd.DataFrame(counts_salmon.X,
                                index=counts_salmon.obs["sample"],
                                columns=counts_salmon.var["gene_ids"]).T
counts_salmon_df.to_csv(OUT_DIR / "gene_counts_salmon.csv")
print(f"✓ Matriz pymportx: {counts_salmon_df.shape[0]} genes × {counts_salmon_df.shape[1]} muestras")
display(counts_salmon_df.head())

> ❓ **Pregunta 4.3:** ¿Por qué es recomendable usar pymportx/tximport en lugar de
> sumar directamente los conteos de los transcritos de cada gen? *(Pista: piensa en genes
> con isoformas de distinta longitud y en cómo afecta la longitud efectiva a los conteos.)*

---
# PARTE V — Comparación de Métodos de Cuantificación

## Actividad 5.1: STAR + featureCounts vs Salmon + pymportx

Comparamos los conteos a nivel de gen obtenidos por ambas rutas. Si ambas funcionan bien,
los conteos deberían estar **altamente correlacionados** (r > 0.95 es lo esperable).

In [ ]:
# ============================================================================
# CELDA 18: Correlación entre métodos
# ============================================================================
from scipy import stats

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
resultados_corr = {}

for i, srr in enumerate(MUESTRAS):
    # Unir ambos métodos por gen (índice = gene_id sin versión)
    fc = counts_fc[srr].copy()
    fc.index = fc.index.str.split(".").str[0]
    fc = fc.groupby(level=0).sum()
    sa = counts_salmon_df[srr].copy()
    sa.index = sa.index.astype(str).str.split(".").str[0]
    sa = sa.groupby(level=0).sum()

    comun = fc.index.intersection(sa.index)
    x, y = fc.loc[comun], sa.loc[comun]
    r_pearson, _ = stats.pearsonr(np.log2(x + 1), np.log2(y + 1))
    r_spearman, _ = stats.spearmanr(x, y)
    resultados_corr[srr] = {"Pearson (log2)": round(r_pearson, 4),
                            "Spearman": round(r_spearman, 4)}

    axes[i].scatter(np.log2(x + 1), np.log2(y + 1), s=4, alpha=0.4)
    axes[i].set_xlabel("featureCounts, log2(conteos+1)")
    axes[i].set_ylabel("Salmon+pymportx, log2(conteos+1)")
    axes[i].set_title(f"{srr} ({MUESTRAS[srr]})\nPearson r = {r_pearson:.3f}")

plt.tight_layout()
plt.savefig(OUT_DIR / "comparacion_metodos.png", dpi=150)
plt.show()
display(pd.DataFrame(resultados_corr).T)

In [ ]:
# ============================================================================
# CELDA 19: Genes más discordantes entre métodos
# ============================================================================

# Razón media entre métodos (en escala log2) para detectar discordancias
srr = "SRR3480371"
fc = counts_fc[srr].copy(); fc.index = fc.index.str.split(".").str[0]; fc = fc.groupby(level=0).sum()
sa = counts_salmon_df[srr].copy(); sa.index = sa.index.astype(str).str.split(".").str[0]; sa = sa.groupby(level=0).sum()
comun = fc.index.intersection(sa.index)
comparacion = pd.DataFrame({"featureCounts": fc.loc[comun], "salmon": sa.loc[comun]})
comparacion["log2FC_ratio"] = np.log2(comparacion["featureCounts"] + 1) - np.log2(comparacion["salmon"] + 1)

# Filtrar genes con expresión razonable y mayor discordancia
activos = comparacion[(comparacion["featureCounts"] > 10) | (comparacion["salmon"] > 10)]
discordantes = activos.reindex(activos["log2FC_ratio"].abs().sort_values(ascending=False).index)
print(f"Genes más discordantes en {srr} (control 1):")
display(discordantes.head(15).round(2))

> ❓ **Preguntas 5.1:**
> 1. ¿Qué nivel de correlación observas entre ambos métodos?
> 2. ¿Hay genes que muestran diferencias importantes entre métodos?
> 3. ¿Cuáles pueden ser las causas de estas diferencias? *(Pista: lecturas multimapeo,
>    pseudogenes, solapamiento de genes, criterio de asignación de featureCounts...)*

---
## Exportación de resultados

In [ ]:
# ============================================================================
# CELDA 20: Comprimir y descargar todos los resultados
# ============================================================================
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/rnaseq_UPF1_results", "zip", OUT_DIR)
print(f"✓ ZIP creado: {zip_path}")
files.download(zip_path)

---
## Recordatorio de entrega

1. **PDF** con las respuestas a todas las preguntas ❓ (bien estructurado, con gráficos
   y tablas cuando sean relevantes, y conclusiones fundamentadas).
2. **Este notebook ejecutado** de principio a fin, con las salidas visibles y tus
   comentarios adicionales.

Se evaluará: claridad de las respuestas, presentación y documentación del código,
funcionalidad (ejecuta sin errores) y completitud.